## Exercises on Dynamic Programming

These activities reinforce the core ideas presented in the DP chapter:: the Bellman expectation and optimality equations, exact policy evaluation by solving a linear system, iterative policy evaluation as a fixed-point iteration, the policy-improvement theorem, value iteration, the existence of multiple optimal policies, the computational efficiency of policy iteration versus brute-force search and versus value iteration, and how discounting interacts with risk in stochastic environments.

### Exercise 3.1 — Exact policy evaluation

Consider the following MDP with two non-terminal states $S_1, S_2$ and one terminal state, evaluated under a fixed policy (one action per state), with $\gamma = 1$:

- In $S_1$: the action leads deterministically to $S_2$ with reward -1.
- In $S_2$: the action leads with probability 0.5 to the terminal state with reward +10, and with probability 0.5 back to $S_1$ with reward -1.

Compute the state values $v_\pi(S_1)$ and $v_\pi(S_2)$.

**Step 1 — Write the Bellman expectation equation** 

$\displaystyle v_\pi(s)=\sum_{s',r}p(s',r\mid s,\pi(s))\,[\,r+\gamma v_\pi(s')\,]$ for each state (with $\gamma=1$, and $v_\pi(\text{terminal})=0$):

$\displaystyle v_\pi(S_1) = 1\cdot\big[-1 + v_\pi(S_2)\big] = -1 + v_\pi(S_2)$      
$\displaystyle v_\pi(S_2) = 0.5\big[10 + 0\big] + 0.5\big[-1 + v_\pi(S_1)\big] = 4.5 + 0.5\,v_\pi(S_1)$        

**Step 2 — Solve the linear system** 

Substitute the first equation into the second:

$\displaystyle v_\pi(S_2) = 4.5 + 0.5\,(-1 + v_\pi(S_2)) = 4.0 + 0.5\,v_\pi(S_2)$.  
$\displaystyle \Rightarrow\; 0.5\,v_\pi(S_2) = 4.0 \;\Rightarrow\; v_\pi(S_2) = 8.$     

Back-substitute: 

$\displaystyle v_\pi(S_1) = -1 + 8 = 7.$

**Key concept**

For a fixed policy the Bellman expectation equations form a linear system with a unique solution, so value functions can be computed exactly when the MDP is small and known. Iterative methods approximate this same solution when a direct solve is impractical.

### Exercise 3.2 — Iterative policy evaluation

For the same MDP, run iterative policy evaluation starting from: 

$\displaystyle V_0(S_1) = 0$        
$\displaystyle V_0(S_2) = 0$

Remember that:

$\displaystyle V_{k+1}(s) \leftarrow \sum_{s',r} p(s',r\mid s,\pi(s))\,[\,r+\gamma V_k(s')\,].$

Perform four sweeps and comment on convergence toward the exact values.

**Step 1 — Update equations**

$\displaystyle V_{k+1}(S_1) = -1 + V_k(S_2)$        
$\displaystyle V_{k+1}(S_2) = 4.5 + 0.5\,V_k(S_1)$

**Step 2 — Iterate**

| sweep $k$ | $V_k(S_1)$ | $V_k(S_2)$ |
|---|---|---|
| 0 | 0 | 0 |
| 1 | $-1+0=-1.0$ | $4.5+0.5(0)=4.5$ |
| 2 | $-1+4.5=3.5$ | $4.5+0.5(-1)=4.0$ |
| 3 | $-1+4.0=3.0$ | $4.5+0.5(3.5)=6.25$ |
| 4 | $-1+6.25=5.25$ | $4.5+0.5(3.0)=6.0$ |

**Step 3 — Convergence** 

The estimates march toward the fixed point (7,8). Each sweep is a contraction toward the true value function.

**Key concept**

Iterative policy evaluation is bootstrapping: it updates each state's estimate from the current estimates of its successors. It converges to the same that solves the linear system, without ever inverting a matrix (the practical route when the state set is large). Remember that the index k counts algorithm sweeps, not environment time steps.

### Exercise 3.3 — Policy improvement by one greedy step

In a state X two actions are available and both give reward 0 on the transition, with $\gamma=0.9$:

- $a_1$: leads deterministically to state Y, with known value $v_\pi(Y)=5$
- $a_2$: leads deterministically to state Z, with known value $v_\pi(Z)=8$

The current policy selects $a_1$ in X.

1. Compute the action values of both actions
2. Apply a policy-improvement step and verify it satisfies the policy-improvement theorem.

**Step 1 — Action values** 

$\displaystyle q_\pi(X,a)=\sum_{s',r}p(s',r\mid X,a)\,[\,r+\gamma v_\pi(s')\,]$ 

We have deterministic transitions and r=0, so:

$\displaystyle q_\pi(X,a_1) = 0 + 0.9\times 5 = 4.5$        
$\displaystyle q_\pi(X,a_2) = 0 + 0.9\times 8 = 7.2$

Under the current policy, $v_\pi(X)=q_\pi(X,a_1)=4.5$.

**Step 2 — Greedy improvement** 

Considering the improvement formula:

$\displaystyle \pi'(X)=\arg\max_a q_\pi(X,a)=a_2$

since 7.2 > 4.5.

**Step 3 — Check the policy-improvement theorem** 

The theorem states that: 

$\displaystyle q_\pi(s,\pi'(s))\ge v_\pi(s)  \forall s, \rightarrow v_{\pi'}\ge v_\pi$

Here

$\displaystyle q_\pi(X,\pi'(X)) = q_\pi(X,a_2) = 7.2 \;\ge\; v_\pi(X) = 4.5,$

so switching to $a_2$ is guaranteed not to worsen (here, to strictly improve) the policy.

**Key concept**

Acting greedily with respect to the current value function yields a policy that is at least as good (the engine behind policy iteration). 

### Exercise 3.4 — Value iteration and policy extraction

Consider states A, B and a terminal goal G (with state value 0), $\gamma=0.9$:

- In A: action "right" -> B (reward 0); action "stay" -> A (reward 0)
- In B: action "right" -> G (reward 1); action "left" -> A (reward 0)

Run value iteration from 

$\displaystyle V_0 = 0$         
$\displaystyle V_{k+1}(s)=\max_a \sum_{s',r}p(s',r\mid s,a)\,[\,r+\gamma V_k(s')\,],$

for three iterations, then extract the greedy optimal policy.

**Step 1 — Iterate.** 

With 

$V_0(A) =0$      
$V_0(B) = 0$        

Iteration 1:    
$\displaystyle V_1(A)=\max(\underbrace{0+0.9\cdot0}_{\text{right}},\ \underbrace{0.9\cdot0}_{\text{stay}})=0.0$     
$\displaystyle V_1(B)=\max(\underbrace{1+0.9\cdot0}_{\text{right}},\ \underbrace{0.9\cdot0}_{\text{left}})=1.0$

Iteration 2:        
$\displaystyle V_2(A)=\max(0+0.9\cdot1,\ 0.9\cdot0)=0.9$        
$\displaystyle V_2(B)=\max(1+0,\ 0.9\cdot0)=1.0$

Iteration 3:        
$\displaystyle V_3(A)=\max(0.9\cdot1,\ 0.9\cdot0.9)=0.9$        
$\displaystyle V_3(B)=\max(1,\ 0.9\cdot0.9)=1.0$

The values have converged: 

$\displaystyle v_*(A)=0.9$          
$\displaystyle v_*(B)=1$

**Step 2 — Extract the greedy policy** 

$\displaystyle \pi_*(s)=\arg\max_a\sum p\,[r+\gamma v_*(s')]$:

- A: "right" gives 0.9 ... 0.9 vs. "stay" 0.9 .... 0.9=0.81$ → **right**.
- B: right gives $1$ vs. left $0.9\cdot0.9=0.81$ → **right**.

So $\pi_*: A\to\text{right},\ B\to\text{right}$ (go $A\to B\to G$).

**Key concept**

Value iteration merges evaluation and improvement into a single $\max$-update — "greedily greedifying" every sweep. The policy is extracted only *once*, at the end, by taking the argmax of the one-step lookahead.

### Exercise 3.5 — How the discount factor changes the optimal action

From a start state the agent chooses once between two actions (each then leads to termination):

- **SOONER:** reward $+1$ at the next step ($R_{t+1}=1$), then terminal.
- **LATER:** reward $0$ at the next step and $+2$ the step after ($R_{t+1}=0,\ R_{t+2}=2$), then terminal.

Find the value of the discount factor $\gamma$ at which the agent is indifferent, and state which action is optimal above and below that threshold.

**Step 1 — Value of each action** (from the return $G_t=\sum_k \gamma^k R_{t+k+1}$):

$\displaystyle v(\text{SOONER}) = \gamma^0\cdot 1 = 1, \qquad v(\text{LATER}) = \gamma^0\cdot 0 + \gamma^1\cdot 2 = 2\gamma.$

**Step 2 — Indifference point.** Set them equal:

$\displaystyle 1 = 2\gamma \;\Rightarrow\; \gamma = \tfrac12.$

**Step 3 — Compare on each side of the threshold.**

- If $\gamma < \tfrac12$: $\ 2\gamma < 1$, so **SOONER** is optimal (a discounted future $+2$ is worth less than an immediate $+1$).
- If $\gamma > \tfrac12$: $\ 2\gamma > 1$, so **LATER** is optimal (patience pays).
- At $\gamma=\tfrac12$: both give value $1$ — the agent is indifferent.

**Key concept**

The optimal policy is **not intrinsic to the rewards alone** — it depends on $\gamma$. Discounting encodes how the agent trades immediate against delayed reward, and changing it can flip which action is best. (This mirrors the chapter's remark that discounting can favour faster-but-riskier policies.)

### Exercise 3.6 — Ties in $q_*$ and multiple optimal policies

In some state $s$, two actions achieve the same optimal action-value:

$q_*(s,a_1) = q_*(s,a_2) = 10,$

while every other action available in $s$ has $q_*(s,a) < 10$. Two candidate policies, $\pi_1$ and $\pi_2$, are identical everywhere except in $s$: $\pi_1(s)=a_1$ and $\pi_2(s)=a_2$.

Show that both $\pi_1$ and $\pi_2$ are optimal, and that they induce the *same* optimal state-value function despite being different policies.

**Step 1 — Value of $s$ under either action.** Since $v_*(s)=\max_a q_*(s,a)$ and the maximum is achieved by both $a_1$ and $a_2$:

$\displaystyle v_*(s) = \max_a q_*(s,a) = 10$

regardless of which of the two tied actions is selected in $s$.

**Step 2 — Both policies are optimal.** By construction, $\pi_1$ and $\pi_2$ act identically at every state other than $s$, and at $s$ both satisfy

$\displaystyle q_*(s,\pi_1(s)) = q_*(s,\pi_2(s)) = v_*(s) = 10,$

which is exactly the condition for a policy to be greedy with respect to $q_*$ — the definition of optimality. So $\pi_1$ and $\pi_2$ are both optimal, and since they agree everywhere else and achieve the same value at $s$, they induce the same state-value function: $v_{\pi_1}=v_{\pi_2}=v_*$, even though $\pi_1\neq\pi_2$.

**Key concept**

An MDP's optimal state-value function $v_*$ is always **unique**, but whenever $q_*$ has a tie at some state, the optimal *policy* is **not**: any way of breaking the tie yields an equally optimal policy. This is exactly why the notes write $\pi_*(s)\in\arg\max_a q_*(s,a)$ using set membership rather than equality.

### Exercise 3.7 — Policy iteration vs. brute-force search

Consider an MDP with $|\mathcal S|=10$ states and $|\mathcal A|=4$ actions. A brute-force search would need to evaluate every deterministic policy. Suppose that, starting from an arbitrary policy, policy iteration reaches an optimal policy after $K=5$ improvement steps.

1. Compute $N_{\text{BF}}=|\mathcal A|^{|\mathcal S|}$ and $N_{\text{PI}}=K+1$.
2. Compute the ratio $N_{\text{BF}}/N_{\text{PI}}$ and comment on it.

**Step 1 — Count the policies.**

$\displaystyle N_{\text{BF}}=|\mathcal A|^{|\mathcal S|}=4^{10}=1{,}048{,}576, \qquad N_{\text{PI}}=K+1=6.$

**Step 2 — Compare.**

$\displaystyle \frac{N_{\text{BF}}}{N_{\text{PI}}}=\frac{1{,}048{,}576}{6}\approx 174{,}763$

Brute-force search would need to examine roughly $175{,}000$ times more policies than policy iteration did to find the same optimal policy.

**Key concept**

Policy iteration does not need to be fast in absolute terms to vastly outperform brute-force search — it wins because every policy it examines is *constructed* using information from the previous one, rather than chosen blindly. As $|\mathcal S|$ grows, $N_{\text{BF}}$ grows exponentially while $K$ typically grows far more slowly, so this gap widens dramatically for larger MDPs.

### Exercise 3.8 — Comparing the cost of Policy Iteration and Value Iteration

For an MDP with $|\mathcal S|=16$ and $|\mathcal A|=4$ (matching FrozenLake), suppose:

- policy iteration needs $K=5$ improvement cycles, each requiring on average $L=40$ policy-evaluation sweeps;
- value iteration needs $M=250$ sweeps in total to converge.

Using the cost estimates from the notes, $C_{\text{PI,cycle}}=O(|\mathcal S|^2(L+|\mathcal A|))$ and $C_{\text{VI,sweep}}=O(|\mathcal S|^2|\mathcal A|)$:

1. Estimate the total cost of each algorithm, $K\cdot C_{\text{PI,cycle}}$ versus $M\cdot C_{\text{VI,sweep}}$.
2. Which algorithm is cheaper here?
3. Find the number of sweeps $M^*$ at which value iteration would become exactly as expensive as policy iteration, and comment.

**Step 1 — Total cost of each algorithm.** With $|\mathcal S|^2=256$:

$\displaystyle K\cdot C_{\text{PI,cycle}} = 5\cdot256\cdot(40+4) = 56{,}320$

$\displaystyle M\cdot C_{\text{VI,sweep}} = 250\cdot256\cdot4 = 256{,}000$

**Step 2 — Compare.** Since $56{,}320 < 256{,}000$, **policy iteration is cheaper** in this instance: its few, expensive evaluate-improve cycles add up to less total work than value iteration's many, individually cheaper sweeps.

**Step 3 — Break-even point.** Setting the two costs equal:

$\displaystyle M^*\cdot|\mathcal S|^2|\mathcal A| = K\cdot|\mathcal S|^2(L+|\mathcal A|) \;\Longrightarrow\; M^*=\frac{K(L+|\mathcal A|)}{|\mathcal A|}=\frac{5\cdot44}{4}=55$

Value iteration would need **fewer than $55$ sweeps** to beat policy iteration here. Since it actually needs $M=250\gg55$, policy iteration wins by a wide margin in this case.

**Key concept**

Neither algorithm dominates the other in general — which is cheaper depends on the *actual* number of sweeps or cycles each needs on the specific MDP, not just on the big-O cost of a single sweep/cycle. The break-even point $M^*=K(L+|\mathcal A|)/|\mathcal A|$ makes this trade-off explicit, and mirrors what the notes observe empirically when timing both algorithms on FrozenLake.

### Exercise 3.9 — Discounting under risk: shortcut vs. safe path

From a state $s$, two actions are available:

- **shortcut**: with probability $p$ leads directly to the goal $G$ with reward $+1$; with probability $1-p$ leads instead to a terminal hole $H$ with reward $-1$ (one step, risky).
- **safe**: leads deterministically to an intermediate state with reward $0$, and from there deterministically to the goal $G$ with reward $+1$ (two steps, no risk).

1. Write $v(\text{shortcut})$ and $v(\text{safe})$ as functions of $p$ and $\gamma$.
2. Find the success probability $p^*(\gamma)$ at which the agent is indifferent between the two actions.
3. Describe how $p^*$ changes as $\gamma\to0$ and as $\gamma\to1$, and relate this to the notes' remark that discounting can favor faster but riskier paths.

**Step 1 — Value of each action.** The shortcut's reward arrives at the very next step, so it is not discounted; the safe path's $+1$ arrives one step later, so it is discounted once:

$\displaystyle v(\text{shortcut}) = p\,(+1) + (1-p)\,(-1) = 2p-1, \qquad v(\text{safe}) = 0 + \gamma\,(+1) = \gamma$

**Step 2 — Indifference point.** Setting the two values equal:

$\displaystyle 2p-1=\gamma \;\Longrightarrow\; p^*(\gamma)=\frac{1+\gamma}{2}$

**Step 3 — Behavior of the threshold.** As $\gamma\to0$ (fully myopic agent), $p^*\to\frac12$: even a coin-flip shortcut is preferred, because the safe path's delayed reward is devalued to almost nothing. As $\gamma\to1$ (fully patient agent), $p^*\to1$: only a shortcut that is virtually guaranteed to succeed can beat the safe path, since its delayed-but-certain reward is now valued at its full worth. So **stronger discounting (smaller $\gamma$) lowers the bar for taking the risky shortcut** — exactly matching the notes' observation that discounting can bias the optimal policy toward faster but riskier routes.

**Key concept**

Discounting does not only make the agent impatient about *time* — it also reshapes its attitude toward *risk*, because a certain-but-delayed reward is devalued relative to an immediate gamble. This is the same mechanism behind the FrozenLake discounting experiment in the notes, made explicit and solvable by hand.